<a href="https://colab.research.google.com/github/AishahAbdulHakeem/skincare-ai/blob/main/04_compatibility_and_product_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 Compatibility & Product Report

This notebook:
- Builds an ingredient pair compatibility scoring system
- Scores pairs by how often they appear in high vs low rated products
- Generates a full product report for any Sephora product:
    ⭐ Rating from reviews
    🧪 Ingredient compatibility with user's routine
    📝 Summary of what reviewers say
- Saves all lookups needed for the app

In [12]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import itertools
from collections import defaultdict
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load Data
Load the final skincare dataset, top ingredients, and review summaries.

In [13]:
# Main data file (from Notebook 02)
skincare = pd.read_pickle("/content/drive/MyDrive/skincare-ai/data/processed/skincare_final.pkl")

# Top ingredients (from Notebook 01)
with open("/content/drive/MyDrive/skincare-ai/data/processed/top_ingredients.pkl", "rb") as f:
    top_ingredients = pickle.load(f)

# Review summaries (from Notebook 02)
with open("/content/drive/MyDrive/skincare-ai/models/review_summaries.pkl", "rb") as f:
    review_summaries = pickle.load(f)

skincare = skincare.dropna(subset=['rating']).reset_index(drop=True)

print("Products:", len(skincare))
print("Columns:", skincare.columns.tolist())
print("Products with summaries:",
      sum(1 for v in review_summaries.values()
          if v != "No reviews available for this product."))

Products: 2351
Columns: ['product_id', 'rating', 'ingredient_list', 'mean_sentiment', 'pct_positive', 'review_count', 'review_summary']
Products with summaries: 2351


In [31]:
water_variants = {
    'water/aqua/eau', 'water\\aqua\\eau', 'aqua',
    'aqua/water', 'eau', 'aqua/water/eau',
    'water (aqua)', 'purified water', 'deionized water',
    'water (aqua, eau)', 'water (aqua/eau)'
}

junk = {'1', '2', '3', '4', '5', ''}

def normalize_ingredients(ing_list):
    normalized = []
    seen = set()
    for ing in ing_list:
        if ing in junk:
            continue
        # Catch ANY water variant — even ones we haven't listed explicitly
        elif ing in water_variants or (
            ('water' in ing or 'aqua' in ing) and len(ing) < 30
        ):
            ing = 'water'
        if ing not in seen:
            normalized.append(ing)
            seen.add(ing)
    return normalized

skincare['ingredient_list'] = skincare['ingredient_list'].apply(normalize_ingredients)

# Verify — check what water-related strings remain
all_ings = [i for ings in skincare['ingredient_list'] for i in ings]
water_related = [i for i in set(all_ings) if 'water' in i or 'aqua' in i]
print("Remaining water-related strings:", water_related)

print("Duplicate check:",
      sum(1 for ings in skincare['ingredient_list']
          if len(ings) != len(set(ings))))

Remaining water-related strings: ['hamamelis virginiana (witch hazel) flower water', 'yellow 6 (ci 15985) <iln35510>rds product name: clarifying lotion 3 (new)   division: cl (clinique)water', 'cucumis sativus (cucumber) fruit water', 'lavan dula angustifolia (lavender) flower water', 'lavandula angustifolia (lavender) flower/leaf/stem water*', 'maris aqua/sea water (maris aqua)', 'cocos nucifera (coconut) water (organic)', 'abeille royale advanced youth watery oil:', 'lavandula angustifolia (lavender) flower water', 'water hamamelis virginiana (witch hazel/hamamélis) water', 'lavandula angustifolia (lavender) flower/leaf/stem water', 'declustered water (-)\\aqua\\eau de-structuree (-)', 'pyrus malus (apple) fruit water', 'citrus aurantium dulcis (orange) fruit water', 'water drench hyaluronic cloud cream hydrating moisturizer:', 'eucalyptus globulus leaf water', 'citrus aurantium bergamia (bergamot) fruit water', 'hamamelis virginiana (witch hazel) water', 'rosmarinus officinalis (ros

## Product Lookups
Build fast dictionaries mapping product ID to ingredients,
rating, review count, name, and brand.

In [32]:

product_ingredients = dict(zip(skincare['product_id'], skincare['ingredient_list']))
product_ratings = dict(zip(skincare['product_id'], skincare['rating']))
product_review_counts = dict(zip(skincare['product_id'], skincare['review_count']))

print("Products in ingredient lookup:", len(product_ingredients))
print("Products in rating lookup:", len(product_ratings))

# Preview — show a sample product's data
sample_id = skincare.iloc[0]['product_id']
print(f"\nSample product: {sample_id}")
print(f"  Rating: {product_ratings[sample_id]:.2f}")
print(f"  Review count: {int(product_review_counts[sample_id])}")
print(f"  Ingredients (first 5): {product_ingredients[sample_id][:5]}")
print(f"  Summary: {review_summaries.get(sample_id, 'N/A')[:100]}...")

Products in ingredient lookup: 2351
Products in rating lookup: 2351

Sample product: P439055
  Rating: 4.54
  Review count: 1319
  Ingredients (first 5): ['collagen (vegan)*', 'water', 'ethylhexyl palmitate', 'oryza sativa (rice) bran extract', 'caprylic/capric triglyceride']
  Summary: For those of you out there who sometimes roll your eyes when you read reviews from 20 year olds atte...


## High vs Low Rated Split
Divide products at the median rating. Ingredient pairs that appear
more in high-rated products will score higher in the compatibility table.

In [33]:
median_rating = skincare['rating'].median()

high_rated = skincare[skincare['rating'] >= median_rating]
low_rated  = skincare[skincare['rating'] <  median_rating]

print("Median rating:", median_rating)
print("High rated products:", len(high_rated))
print("Low rated products:", len(low_rated))

Median rating: 4.3111
High rated products: 1176
Low rated products: 1175


## Ingredient Pair Scoring
For every pair of ingredients, compute how frequently they co-occur
in high-rated vs low-rated products. A ratio above 1 means the pair
tends to appear in better-rated products.

In [34]:
def count_cooccurrences(df, top_ingredients):
    counts = defaultdict(int)
    for ing_list in df['ingredient_list']:
        present = [i for i in ing_list if i in top_ingredients]
        for pair in itertools.combinations(sorted(present), 2):
            counts[pair] += 1
    return counts

print("Building co-occurrence counts...")
high_counts = count_cooccurrences(high_rated, top_ingredients)
low_counts  = count_cooccurrences(low_rated,  top_ingredients)

n_high = len(high_rated)
n_low  = len(low_rated)

all_pairs = set(high_counts.keys()) | set(low_counts.keys())

pair_scores = {}
for pair in all_pairs:
    high_rate = high_counts.get(pair, 0) / n_high
    low_rate  = low_counts.get(pair, 0)  / n_low
    pair_scores[pair] = (high_rate + 1e-6) / (low_rate + 1e-6)

print("Total pairs in lookup table:", len(pair_scores))

Building co-occurrence counts...
Total pairs in lookup table: 22062


## Compatibility Function
Given a user's routine ingredients and a selected Sephora product,
score all cross-pairs and return a risk label:
- Low Risk ✅  — pairs tend to appear in high-rated products
- Moderate Risk ⚠️  — mixed signal
- High Risk ❌  — pairs tend to appear in low-rated products

In [35]:
def check_compatibility(routine_ingredients, product_id,
                         product_ingredients, pair_scores, top_ingredients):
    """
    routine_ingredients : list of ingredients the user already uses
    product_id          : Sephora product_id the user wants to buy
    """

    if product_id not in product_ingredients:
        return None, f"Product {product_id} not found in database", []

    new_product_ings = product_ingredients[product_id]

    # Filter both to top ingredients only
    routine_top = [i for i in routine_ingredients if i in top_ingredients]
    new_top     = [i for i in new_product_ings  if i in top_ingredients]

    if not routine_top:
        return None, "No recognized ingredients found in your routine", []
    if not new_top:
        return None, "No recognized ingredients found in selected product", []

    # Score only cross pairs (routine ingredient × new product ingredient)
    cross_pairs = []
    for r_ing in routine_top:
        for n_ing in new_top:
            pair = tuple(sorted([r_ing, n_ing]))
            cross_pairs.append(pair)

    scores = [(pair, pair_scores.get(pair, 1.0)) for pair in cross_pairs]
    avg_score = np.mean([s for _, s in scores])

    # Worst flagged pairs
    flagged = sorted(
        [(p, s) for p, s in scores if s < 0.85],
        key=lambda x: x[1]
    )[:5]

    # Risk label
    if avg_score >= 1.2:
        label = "Low Risk ✅"
    elif avg_score >= 0.9:
        label = "Moderate Risk ⚠️"
    else:
        label = "High Risk ❌"

    return round(avg_score, 4), label, flagged

## Product Report
Combines compatibility score, product rating, and review summary
into a single structured output for the app.

In [37]:
def get_product_report(routine_ingredients, product_id,
                        product_ingredients, product_ratings,
                        product_review_counts, review_summaries,
                        pair_scores, top_ingredients):
    """
    Returns the full app output for a given product:
    - Compatibility score + risk label + flagged pairs
    - Product rating from reviews
    - Review summary
    """

    print("=" * 55)
    name  = product_names.get(product_id, product_id)
    brand = brand_names.get(product_id, "")
    print(f"  {brand} — {name}")
    print("=" * 55)

    # --- Rating ---
    rating       = product_ratings.get(product_id, None)
    review_count = product_review_counts.get(product_id, 0)

    if rating:
        print(f"\n⭐ Rating:        {rating:.2f} / 5  "
              f"({int(review_count):,} reviews)")
    else:
        print("\n⭐ Rating:        Not available")

    # --- Compatibility ---
    score, label, flagged = check_compatibility(
        routine_ingredients, product_id,
        product_ingredients, pair_scores, top_ingredients
    )

    print(f"\n🧪 Compatibility: {label}  (score: {score})")

    if flagged:
        print("\n   ⚠️  Flagged ingredient combinations:")
        for pair, s in flagged:
            print(f"      • {pair[0]}  +  {pair[1]}")
    else:
        print("\n   ✅ No problematic ingredient combinations found.")

    # --- Review Summary ---
    summary = review_summaries.get(product_id, "No reviews available.")
    print(f"\n📝 What people say:\n   {summary}")
    print("=" * 55)

## Testing
Run the product report on sample products to validate the output.

In [38]:
# Simulate a user's current routine ingredients
my_routine = [
    "niacinamide", "glycerin", "hyaluronic acid",
    "dimethicone", "cetyl alcohol", "water"
]

# User selects a product from the Sephora database
target_product_id = skincare.iloc[10]['product_id']

get_product_report(
    my_routine,
    target_product_id,
    product_ingredients,
    product_ratings,
    product_review_counts,
    review_summaries,
    pair_scores,
    top_ingredients
)

  Algenist — Overnight Restorative Cream

⭐ Rating:        4.38 / 5  (239 reviews)

🧪 Compatibility: Low Risk ✅  (score: 6.7035)

   ⚠️  Flagged ingredient combinations:
      • algae extract  +  hyaluronic acid
      • glyceryl stearate se  +  hyaluronic acid
      • cetyl alcohol  +  fragrance
      • cetyl alcohol  +  polysorbate 20
      • cetyl alcohol  +  palmitoyl tetrapeptide-7

📝 What people say:
   I sure do wish these products were less expensive, but I’m willing to pay based on the results I’m getting, my face hasn’t looked or felt this good since I turned 40 a couple years ago. I really loved how it made my skin look and feel, but the scent was too much for me. There are times when I wish I could just put my makeup on right after I wake up, because I think my normal-dry skin looks and feels so much better after applying this cream.


In [39]:
# See how output varies across products
for i in [0, 25, 100]:
    pid = skincare.iloc[i]['product_id']
    get_product_report(
        my_routine, pid,
        product_ingredients, product_ratings,
        product_review_counts, review_summaries,
        pair_scores, top_ingredients
    )
    print()

  Algenist — GENIUS Sleeping Collagen Moisturizer

⭐ Rating:        4.54 / 5  (1,319 reviews)

🧪 Compatibility: Low Risk ✅  (score: 39.3228)

   ⚠️  Flagged ingredient combinations:
      • niacinamide  +  oryza sativa (rice) bran extract
      • ammonium acryloyldimethyltaurate/vp copolymer  +  hyaluronic acid
      • dimethicone  +  hyaluronic acid
      • cetyl alcohol  +  ethylhexyl palmitate
      • ethylhexyl palmitate  +  hyaluronic acid

📝 What people say:
   For those of you out there who sometimes roll your eyes when you read reviews from 20 year olds attesting to the “miracles“ some products do for their wrinkles, let me tell you as a woman over 50, that this really does a wonderful job, so much so that I will probably bite the bullet and shell out the big bucks to purchase once my samples are done. My skin did not see any benefits by using this product.I received this product complimentary for sampling purposes but all opinions are my own. If...

  Algenist — ELEVATE Advanc

In [40]:
import os
os.makedirs("/content/drive/MyDrive/skincare-ai/models", exist_ok=True)

with open("/content/drive/MyDrive/skincare-ai/models/pair_scores.pkl", "wb") as f:
    pickle.dump(pair_scores, f)

with open("/content/drive/MyDrive/skincare-ai/models/product_ingredients.pkl", "wb") as f:
    pickle.dump(product_ingredients, f)

with open("/content/drive/MyDrive/skincare-ai/models/product_ratings.pkl", "wb") as f:
    pickle.dump(product_ratings, f)

with open("/content/drive/MyDrive/skincare-ai/models/product_review_counts.pkl", "wb") as f:
    pickle.dump(product_review_counts, f)

with open("/content/drive/MyDrive/skincare-ai/models/product_names.pkl", "wb") as f:
    pickle.dump(product_names, f)

with open("/content/drive/MyDrive/skincare-ai/models/brand_names.pkl", "wb") as f:
    pickle.dump(brand_names, f)

print("Saved:")
print("  pair_scores.pkl")
print("  product_ingredients.pkl")
print("  product_ratings.pkl")
print("  product_review_counts.pkl")
print("  product_names.pkl")
print("  brand_names.pkl")

Saved:
  pair_scores.pkl
  product_ingredients.pkl
  product_ratings.pkl
  product_review_counts.pkl
  product_names.pkl
  brand_names.pkl
